# KIP - Visual Inspection of Angle Grinder Components

**Course:** wbk Seminar - Kreislauffabrik (KIT)
**Goal:** Real-time visual inspection of angle-grinder components on an
NVIDIA Jetson Orin Nano (8 GB shared RAM) with an external camera.

The notebook follows the four work packages (AP1-AP4) of the project and
the CRISP-DM cycle.

| AP | Task | Section |
|----|------|---------|
| AP1 | Data understanding & preparation | 1 |
| AP2 | Component classification & instance segmentation | 2 |
| AP3 | Defect detection on close-ups | 3 |
| AP4 | Edge deployment (Jetson Orin Nano 8 GB) | 4 |

The notebook is fully convertible to a Python script
(`jupyter nbconvert --to script kip_inspection.ipynb`); cells tagged
`skip-export` (training, plotting) are skipped on the edge device, while
the `export-as-script` cell at the bottom contains the production
real-time loop and is the entry point on the Jetson.


## 0. Setup & Imports

In [ ]:
"""Install dependencies (skip on Jetson - use the JetPack-provided wheels)."""
%pip install -q -r ../requirements.txt


: 

In [ ]:
"""Standard imports + global config."""
from __future__ import annotations

import json
import os
import random
import shutil
import time
from pathlib import Path
from typing import Iterable

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Torch {torch.__version__} on device: {DEVICE}")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110


In [ ]:
"""Project-wide path constants."""
PROJECT_ROOT = Path("/Users/justusschenk/Projects/KIP")
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

for d in (MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "synth":   DATA_DIR / "synth_Daten",
    "real_v1": DATA_DIR / "object_segmentation_real_1088",
    "real_v3": DATA_DIR / "object_segmentation_real_v3_1088",   # primary real data
}

CLASS_NAMES = [
    "anti-vibration_handle", "bearing_plate", "bevel_gear_drive",
    "bevel_gear_spindle", "gearbox_housing", "intermediate_gearbox",
    "motor_housing", "shaft", "wheel_guard",
]
NUM_CLASSES = len(CLASS_NAMES)

for name, p in DATASETS.items():
    print(f"  {name:8s} -> {p}  exists={p.exists()}")


## 1. AP1 - Data Understanding & Preparation

We hold three datasets:

* **Dataset A** - synthetic renders (4400 train / 1100 val).
* **Dataset B** - first real capture (713 / 178).
* **Dataset B v3** - newest real capture (771 / 191) with extra
  `bevel_gear_spindle` close-ups; **this is our primary real source**.

> **Dataset C (defect close-ups) is not yet available.** AP3 below
> contains a stub loader and an unsupervised approach (PatchCore) that
> can bootstrap defect detection without labelled defects.


In [ ]:
"""Load and visualise per-class label counts from each dataset."""
def load_label_counts(dataset_dir: Path) -> pd.DataFrame:
    """Return a DataFrame from a dataset's `label_counts.csv` if present."""
    csv = dataset_dir / "label_counts.csv"
    if not csv.exists():
        return pd.DataFrame()
    return pd.read_csv(csv)

label_dfs = {name: load_label_counts(p) for name, p in DATASETS.items()}
for name, df in label_dfs.items():
    print(f"\n=== {name} ===")
    print(df.head(20))


In [ ]:
"""Bar plot of the class distribution per dataset."""
fig, axes = plt.subplots(1, len(label_dfs), figsize=(18, 4), sharey=False)
for ax, (name, df) in zip(axes, label_dfs.items()):
    if df.empty:
        ax.text(0.5, 0.5, f"no label_counts.csv\nin {name}", ha="center", va="center")
        ax.set_axis_off()
        continue
    # Heuristic: detect class-name + count columns regardless of exact header.
    name_col = next((c for c in df.columns if "class" in c.lower() or "name" in c.lower()), df.columns[0])
    count_col = next((c for c in df.columns if "count" in c.lower() or "total" in c.lower()), df.columns[-1])
    sns.barplot(data=df, x=name_col, y=count_col, ax=ax, color="steelblue")
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=60)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
"""Sanity check: image counts and a 5-sample resolution probe per dataset."""
def dataset_summary(dataset_dir: Path) -> dict:
    """Count images / labels and probe a few image sizes."""
    img_train = sorted((dataset_dir / "images" / "train").glob("*"))
    img_val   = sorted((dataset_dir / "images" / "val").glob("*"))
    lbl_train = sorted((dataset_dir / "labels" / "train").glob("*.txt"))
    lbl_val   = sorted((dataset_dir / "labels" / "val").glob("*.txt"))
    polygon_count = 0
    for txt in lbl_train[:200]:                 # cheap sample
        polygon_count += sum(1 for _ in txt.read_text().splitlines() if _.strip())
    sample = random.sample(img_train, k=min(5, len(img_train)))
    sizes = []
    for s in sample:
        im = cv2.imread(str(s))
        if im is not None:
            sizes.append(im.shape[:2])
    return {
        "train_images": len(img_train),
        "val_images":   len(img_val),
        "train_labels": len(lbl_train),
        "val_labels":   len(lbl_val),
        "approx_polygons_in_first200_train_labels": polygon_count,
        "sample_sizes": sizes,
    }

for name, p in DATASETS.items():
    if not p.exists():
        print(f"{name}: MISSING")
        continue
    print(f"\n--- {name} ---")
    for k, v in dataset_summary(p).items():
        print(f"  {k}: {v}")


In [ ]:
"""Visualise three random samples per dataset with polygon overlays."""
def yolo_polygons(label_path: Path) -> list[tuple[int, np.ndarray]]:
    """Parse a YOLO-seg .txt and return [(class_id, Nx2 normalised polygon), ...]."""
    polys = []
    if not label_path.exists():
        return polys
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(parts[0])
        coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        polys.append((cls, coords))
    return polys

def show_samples(dataset_dir: Path, k: int = 3, title: str = "") -> None:
    """Plot `k` random images with polygon overlays from a YOLO-seg dataset."""
    img_dir = dataset_dir / "images" / "train"
    lbl_dir = dataset_dir / "labels" / "train"
    images = sorted(img_dir.glob("*"))
    if not images:
        print(f"{title}: no images")
        return
    picks = random.sample(images, k=min(k, len(images)))
    fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 5))
    if len(picks) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, picks):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for cls, poly in yolo_polygons(lbl_dir / (img_path.stem + ".txt")):
            pts = (poly * np.array([w, h])).astype(np.int32)
            cv2.polylines(img, [pts], isClosed=True, color=(255, 0, 0), thickness=2)
            cv2.putText(img, CLASS_NAMES[cls], tuple(pts[0]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        ax.imshow(img)
        ax.set_axis_off()
        ax.set_title(img_path.name, fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

for name, p in DATASETS.items():
    if p.exists():
        show_samples(p, k=3, title=name)


In [ ]:
"""Tool-ID-based val/test split (true move, idempotent).

The data was captured per physical angle-grinder (tool01..tool99). The
provided val/ set already contains only tools that NEVER appear in train
(currently tool03 + tool10), so val/ is a clean tool-level OOS split.

To obtain a held-out *test* set without leaking the same physical tools
across val and test, we MOVE all images+labels of `test_tools` from
val/ into a new test/ folder.

Default policy:  val -> tool03,  test -> tool10
"""
import re

TOOL_RE = re.compile(r"(tool\d+)_")

def _tool_of(name: str) -> str | None:
    """Extract the tool-ID prefix (e.g. 'tool10') from a filename, or None."""
    m = TOOL_RE.match(name)
    return m.group(1) if m else None

def create_tool_based_split(dataset_path: Path,
                            test_tools: tuple[str, ...] = ("tool10",),
                            val_tools: tuple[str, ...] = ("tool03",)
                            ) -> tuple[int, int]:
    """Move images+labels of `test_tools` from val/ to test/. Returns (#moved, #val_remaining).

    Idempotent: re-runs are no-ops once the test/ folder already contains
    the test_tools files.
    """
    img_val  = dataset_path / "images" / "val"
    lbl_val  = dataset_path / "labels" / "val"
    img_test = dataset_path / "images" / "test"
    lbl_test = dataset_path / "labels" / "test"
    img_test.mkdir(parents=True, exist_ok=True)
    lbl_test.mkdir(parents=True, exist_ok=True)

    test_set = set(test_tools)
    moved = 0
    for img in sorted(img_val.glob("*")):
        if _tool_of(img.name) in test_set:
            shutil.move(str(img), str(img_test / img.name))
            lbl = lbl_val / (img.stem + ".txt")
            if lbl.exists():
                shutil.move(str(lbl), str(lbl_test / lbl.name))
            moved += 1

    # Drop stale ultralytics label caches (regenerated on next val/test).
    for cache in (lbl_val.parent / "val.cache",
                  lbl_val.parent / "test.cache"):
        if cache.exists():
            cache.unlink()

    val_remaining = sum(1 for _ in img_val.glob("*"))
    val_tools_actual = sorted({_tool_of(p.name) for p in img_val.glob("*")} - {None})
    test_tools_actual = sorted({_tool_of(p.name) for p in img_test.glob("*")} - {None})
    print(f"[{dataset_path.name}] moved {moved} -> test/  |  "
          f"val={val_remaining} ({val_tools_actual})  |  "
          f"test={sum(1 for _ in img_test.glob('*'))} ({test_tools_actual})")
    return moved, val_remaining


for name in ("real_v1", "real_v3"):
    if DATASETS[name].exists():
        create_tool_based_split(DATASETS[name],
                                test_tools=("tool10",),
                                val_tools=("tool03",))


In [ ]:
"""Write an eval data.yaml pointing val: -> images/test/ for OOS evaluation.

The original data.yaml inside each dataset has train->images/train and
val->images/val. After `create_tool_based_split`, that val/ contains
ONLY tool03 (used for model selection). The eval YAML below redirects
ultralytics's `val:` key to images/test/ (tool10) so `model.val()`
runs on truly held-out physical hardware - the gold-standard OOS metric.
"""
def write_eval_yaml(dataset_path: Path, out_path: Path) -> Path:
    """Write a tiny YAML pointing val: -> images/test for OOS evaluation."""
    yaml_text = (
        f"path: {dataset_path}\n"
        f"train: images/train\n"
        f"val: images/test\n"      # ultralytics treats `val:` as the eval split
        f"nc: {NUM_CLASSES}\n"
        "names:\n"
        + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES))
    )
    out_path.write_text(yaml_text)
    return out_path

EVAL_YAML_REAL = write_eval_yaml(DATASETS["real_v3"],
                                  RESULTS_DIR / "eval_real_v3_test.yaml")
print("OOS eval YAML ->", EVAL_YAML_REAL)


## 2. AP2 - Component Classification & Instance Segmentation

We use **YOLOv11n-seg** (~6 MB) as the production backbone because it
hits ~10-15 FPS at 640x640 on the Jetson Orin Nano with TensorRT INT8.
A `MODEL_VARIANT` flag lets us swap in `yolo11s-seg.pt` (~22 MB) for an
offline quality benchmark.

Three datamix experiments:

| Exp | Train data | Eval data |
|-----|------------|-----------|
| A | Dataset A (synth) only | real_v3 / test |
| B | Dataset B v3 (real)    | real_v3 / test |
| C | A pretrain -> fine-tune on B v3 | real_v3 / test |


In [ ]:
"""YOLO config flags - tweak EPOCHS for a real run, leave low for smoke test."""
from ultralytics import YOLO

MODEL_VARIANT = "yolo11n-seg.pt"        # production. set to yolo11s-seg.pt for quality study
IMG_SIZE      = 640                     # Jetson-aligned; native dataset is 1088
BATCH         = 16
EPOCHS        = 5                       # production: 50-100
PROJECT       = str(RESULTS_DIR / "yolo_runs")


In [ ]:
"""Generic YOLO-seg training helper used by all three datamix experiments."""
def train_yolo(data_yaml: Path, model_variant: str, epochs: int,
               imgsz: int = 640, batch: int = 16, name: str = "exp",
               weights: str | None = None) -> YOLO:
    """Train (or fine-tune) a YOLO seg model and return the trained `YOLO` handle.

    `weights` overrides `model_variant` and is used for transfer learning
    (Exp C continues from the synth-trained checkpoint).
    """
    init = weights or model_variant
    model = YOLO(init)
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=DEVICE,
        project=PROJECT,
        name=name,
        exist_ok=True,
        seed=SEED,
        verbose=False,
    )
    return model


def evaluate_yolo(model: YOLO, eval_yaml: Path) -> dict:
    """Run validation against a held-out YAML and return a flat metrics dict."""
    res = model.val(data=str(eval_yaml), imgsz=IMG_SIZE, batch=BATCH,
                    device=DEVICE, verbose=False, split="val")
    box = res.box
    return {
        "mAP50":       float(box.map50),
        "mAP50_95":    float(box.map),
        "precision":   float(box.mp),
        "recall":      float(box.mr),
        "f1_macro":    float((2 * box.mp * box.mr) / (box.mp + box.mr + 1e-9)),
    }


In [ ]:
"""Exp A: train on synthetic only, evaluate on the real test split."""
metrics: dict[str, dict] = {}

synth_yaml = DATASETS["synth"] / "data.yaml"
model_a = train_yolo(synth_yaml, MODEL_VARIANT, EPOCHS, IMG_SIZE, BATCH, name="A_synth_only")
metrics["A_synth_only"] = evaluate_yolo(model_a, EVAL_YAML_REAL)
print("Exp A:", metrics["A_synth_only"])


In [ ]:
"""Exp B: train on real v3, evaluate on the real test split."""
real_yaml = DATASETS["real_v3"] / "data.yaml"
model_b = train_yolo(real_yaml, MODEL_VARIANT, EPOCHS, IMG_SIZE, BATCH, name="B_real_only")
metrics["B_real_only"] = evaluate_yolo(model_b, EVAL_YAML_REAL)
print("Exp B:", metrics["B_real_only"])


In [ ]:
"""Exp C: synth pretrain -> fine-tune on real v3 (transfer learning)."""
synth_weights = Path(PROJECT) / "A_synth_only" / "weights" / "best.pt"
model_c = train_yolo(real_yaml, MODEL_VARIANT, EPOCHS, IMG_SIZE, BATCH,
                     name="C_synth_pretrain_real_finetune",
                     weights=str(synth_weights) if synth_weights.exists() else None)
metrics["C_transfer"] = evaluate_yolo(model_c, EVAL_YAML_REAL)
print("Exp C:", metrics["C_transfer"])

(RESULTS_DIR / "exp_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Metrics saved to", RESULTS_DIR / "exp_metrics.json")


In [ ]:
"""Side-by-side comparison of the three datamix experiments."""
metrics_path = RESULTS_DIR / "exp_metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text())).T
    df.plot(kind="bar", figsize=(10, 4), rot=15)
    plt.title("YOLOv11n-seg datamix comparison (real test set)")
    plt.ylabel("score")
    plt.tight_layout()
    plt.show()
    print(df.round(3))
else:
    print("Run experiments A/B/C first.")


In [ ]:
"""Qualitative segmentation samples from the best model on real images."""
def show_predictions(model: YOLO, image_dir: Path, k: int = 4) -> None:
    """Plot `k` random predictions (input + overlay) from a directory of images."""
    images = sorted(image_dir.glob("*"))
    if not images:
        print("no images to predict on")
        return
    picks = random.sample(images, k=min(k, len(images)))
    fig, axes = plt.subplots(2, len(picks), figsize=(4 * len(picks), 8))
    for col, img_path in enumerate(picks):
        res = model.predict(source=str(img_path), imgsz=IMG_SIZE,
                            device=DEVICE, verbose=False)[0]
        original = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        overlay = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        axes[0, col].imshow(original); axes[0, col].set_axis_off()
        axes[1, col].imshow(overlay);  axes[1, col].set_axis_off()
    axes[0, 0].set_title("input", loc="left")
    axes[1, 0].set_title("predicted overlay", loc="left")
    plt.tight_layout(); plt.show()

best_model = locals().get("model_c") or locals().get("model_b") or locals().get("model_a")
if best_model is not None:
    show_predictions(best_model, DATASETS["real_v3"] / "images" / "test", k=4)


## 3. AP3 - Defect Detection on Close-ups

> **Dataset C (defect close-ups) is not yet available.**
> We therefore design AP3 around **PatchCore** (an unsupervised
> patch-based anomaly detector) so that we can:
>
> 1. learn the *normal* appearance of each component from the real data
>    we already have (B v3),
> 2. plug in real defect close-ups (Dataset C) for evaluation as soon
>    as they arrive, *without* retraining,
> 3. fall back to a pure-PyTorch implementation if `anomalib` is not
>    available on the target machine.
>
> **Alternatives considered:**
> - **Few-shot CLIP / DINOv2 classification** - simple but needs a few
>   labelled defects per class; deferrable until Dataset C is delivered.
> - **Synthetic defect generation (CutPaste / NSA)** - cheap data but
>   covers only a narrow defect distribution; useful as data augmentation.


In [ ]:
"""Use the trained YOLO model to extract per-class component crops."""
def extract_component_crops(yolo_model, image_dir: Path, class_id: int,
                             output_dir: Path, padding: int = 10,
                             imgsz: int = 640, max_crops: int = 200) -> int:
    """Crop instances of `class_id` predicted by `yolo_model`; returns count."""
    output_dir.mkdir(parents=True, exist_ok=True)
    images = sorted(image_dir.glob("*"))
    saved = 0
    for img_path in tqdm(images, desc=f"crops/{CLASS_NAMES[class_id]}"):
        if saved >= max_crops:
            break
        res = yolo_model.predict(source=str(img_path), imgsz=imgsz,
                                 device=DEVICE, verbose=False)[0]
        if res.boxes is None or len(res.boxes) == 0:
            continue
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        for i, cls in enumerate(res.boxes.cls.cpu().numpy().astype(int)):
            if cls != class_id:
                continue
            x1, y1, x2, y2 = res.boxes.xyxy[i].cpu().numpy().astype(int)
            x1 = max(0, x1 - padding); y1 = max(0, y1 - padding)
            x2 = min(w, x2 + padding); y2 = min(h, y2 + padding)
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            cv2.imwrite(str(output_dir / f"{img_path.stem}_{i}.png"), crop)
            saved += 1
            if saved >= max_crops:
                break
    return saved

# Demo: extract motor_housing crops from real_v3 train (only if a model exists).
if "model_c" in locals() or "model_b" in locals():
    yolo_for_crops = locals().get("model_c") or locals().get("model_b")
    crops_dir = DATA_DIR / "patchcore_normal" / "motor_housing"
    n = extract_component_crops(yolo_for_crops,
                                DATASETS["real_v3"] / "images" / "train",
                                class_id=CLASS_NAMES.index("motor_housing"),
                                output_dir=crops_dir,
                                padding=10, max_crops=80)
    print(f"saved {n} crops to {crops_dir}")


In [ ]:
"""Train PatchCore via anomalib if available, else fall back to a PyTorch impl."""
def train_patchcore(normal_crops_dir: Path,
                    backbone: str = "resnet18",
                    layers: tuple[str, ...] = ("layer2", "layer3"),
                    coreset_ratio: float = 0.1):
    """Train a PatchCore detector on normal crops; returns the model object."""
    try:
        from anomalib.models.image import Patchcore
        from anomalib.data.image.folder import Folder
        from anomalib.engine import Engine

        datamodule = Folder(
            name="kip_normal",
            root=str(normal_crops_dir.parent),
            normal_dir=normal_crops_dir.name,
            image_size=(224, 224),
            train_batch_size=8,
            eval_batch_size=8,
            num_workers=2,
        )
        model = Patchcore(backbone=backbone, layers=list(layers),
                          coreset_sampling_ratio=coreset_ratio)
        engine = Engine(accelerator="auto", devices=1, max_epochs=1)
        engine.fit(model=model, datamodule=datamodule)
        return ("anomalib", model, engine)
    except Exception as exc:  # noqa: BLE001
        print(f"anomalib unavailable ({exc!s:.80}); using fallback PatchCore.")
        return _train_patchcore_fallback(normal_crops_dir, backbone)


def _train_patchcore_fallback(normal_crops_dir: Path, backbone: str = "resnet18"):
    """Minimal nearest-neighbour memory-bank PatchCore using torchvision features."""
    import torchvision.models as tvm
    from torchvision.transforms import functional as TF

    if backbone == "resnet18":
        net = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
    else:
        net = tvm.wide_resnet50_2(weights=tvm.Wide_ResNet50_2_Weights.DEFAULT)
    extractor = torch.nn.Sequential(*list(net.children())[:7]).eval().to(DEVICE)

    feats = []
    for img_path in sorted(normal_crops_dir.glob("*.png")):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        t = TF.to_tensor(img)
        t = TF.normalize(t, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        with torch.no_grad():
            f = extractor(t.unsqueeze(0).to(DEVICE)).squeeze(0)
        f = f.permute(1, 2, 0).reshape(-1, f.shape[0]).cpu()
        feats.append(f)
    if not feats:
        raise RuntimeError(f"No crops in {normal_crops_dir}")
    bank = torch.cat(feats, dim=0)
    # Coreset subsample to keep memory small.
    idx = torch.randperm(bank.shape[0])[: max(1, bank.shape[0] // 10)]
    bank = bank[idx]
    return ("fallback", {"bank": bank, "extractor": extractor})


def load_dataset_c(path: Path):
    """Stub loader for Dataset C (defect close-ups). Errors loudly when missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset C is not on disk at {path}. "
            "Once it is delivered, organise it as: <root>/<class>/<defect_or_ok>/*.png "
            "and re-run this notebook from AP3 onwards."
        )
    return sorted(path.rglob("*.png"))


In [ ]:
"""Anomaly inference: score a single image and return (score, map, is_defect)."""
def detect_defect(image: np.ndarray, patchcore_model, threshold: float = 0.5):
    """Return (anomaly_score, anomaly_map, is_defect) for one BGR image."""
    backend, payload = patchcore_model[0], patchcore_model[1]
    if backend == "anomalib":
        from torchvision.transforms import functional as TF
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (224, 224))
        t = TF.to_tensor(rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out = payload(t)
        score = float(out["pred_score"].item())
        amap = out["anomaly_map"].squeeze().detach().cpu().numpy()
        return score, amap, score > threshold

    # Fallback nearest-neighbour scoring.
    from torchvision.transforms import functional as TF
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (224, 224))
    t = TF.to_tensor(rgb)
    t = TF.normalize(t, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        f = payload["extractor"](t).squeeze(0)
    h, w = f.shape[1], f.shape[2]
    f = f.permute(1, 2, 0).reshape(-1, f.shape[0]).cpu()
    bank = payload["bank"]
    # Cosine distance to nearest bank vector for every spatial token.
    f = torch.nn.functional.normalize(f, dim=1)
    bn = torch.nn.functional.normalize(bank, dim=1)
    sim = f @ bn.T
    dist = 1.0 - sim.max(dim=1).values
    amap = dist.reshape(h, w).numpy()
    score = float(dist.max())
    return score, amap, score > threshold


In [ ]:
"""Demo: train PatchCore on the motor_housing crops then score a couple of them."""
crops_dir = DATA_DIR / "patchcore_normal" / "motor_housing"
if crops_dir.exists() and any(crops_dir.iterdir()):
    pc = train_patchcore(crops_dir, backbone="resnet18")
    sample = random.sample(list(crops_dir.glob("*.png")), k=min(3, len(list(crops_dir.glob("*.png")))))
    fig, axes = plt.subplots(2, len(sample), figsize=(4 * len(sample), 6))
    for col, p in enumerate(sample):
        img = cv2.imread(str(p))
        score, amap, is_def = detect_defect(img, pc, threshold=0.5)
        axes[0, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0, col].set_axis_off()
        axes[0, col].set_title(f"score={score:.2f} defect={is_def}")
        axes[1, col].imshow(amap, cmap="inferno"); axes[1, col].set_axis_off()
    plt.tight_layout(); plt.show()
else:
    print(f"No crops at {crops_dir} - run the AP3 crop extraction cell first.")


## 4. AP4 - Edge Deployment (Jetson Orin Nano 8 GB)

Two-stage pipeline:

```
camera frame  ->  YOLOv11n-seg (TRT INT8)  ->  per-component crop
                                                ->  PatchCore (TRT FP16)
                                                ->  overlay + display
```

**Memory budget** (RAM is shared with the GPU on the Orin Nano):

| Component | Footprint |
|-----------|-----------|
| YOLOv11n-seg INT8 engine    | ~10 MB |
| PatchCore (resnet18) + bank | ~80 MB |
| Camera + OpenCV buffers     | ~150 MB |
| **Total**                   | **< 0.5 GB** (well under the 2 GB cap) |

Throughput target: **>= 10 FPS at 640x640** with TensorRT INT8.


In [ ]:
"""Export the trained YOLO seg model to ONNX for downstream TRT conversion."""
def export_yolo_onnx(model: "YOLO", out_path: Path, imgsz: int = 640) -> Path:
    """Export ultralytics YOLO seg to ONNX with the flags TensorRT prefers."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    onnx_path = model.export(format="onnx", imgsz=imgsz, half=True,
                             simplify=True, opset=12)
    onnx_path = Path(onnx_path)
    final = out_path
    if onnx_path.resolve() != final.resolve():
        shutil.copy2(onnx_path, final)
    print(f"YOLO ONNX -> {final}")
    return final

if "model_c" in locals() or "model_b" in locals() or "model_a" in locals():
    src_model = locals().get("model_c") or locals().get("model_b") or locals().get("model_a")
    YOLO_ONNX = export_yolo_onnx(src_model, MODELS_DIR / "yolo11n_seg.onnx", imgsz=IMG_SIZE)
else:
    print("No trained YOLO model in scope - run AP2 first.")


In [ ]:
"""Export PatchCore feature extractor to ONNX (or TorchScript fallback)."""
def export_patchcore(patchcore_model, out_dir: Path) -> Path:
    """Export the feature extractor + persist the memory bank."""
    out_dir.mkdir(parents=True, exist_ok=True)
    backend, payload = patchcore_model[0], patchcore_model[1]
    if backend == "fallback":
        extractor = payload["extractor"]
        bank = payload["bank"]
        dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
        onnx_path = out_dir / "patchcore_resnet18.onnx"
        torch.onnx.export(extractor, dummy, str(onnx_path),
                          input_names=["input"], output_names=["features"],
                          opset_version=12,
                          dynamic_axes={"input": {0: "N"}, "features": {0: "N"}})
        torch.save(bank, out_dir / "patchcore_bank.pt")
        print("PatchCore ONNX ->", onnx_path)
        return onnx_path
    # anomalib path: TorchScript is the safe export
    ts_path = out_dir / "patchcore_anomalib.ts"
    try:
        torch.jit.save(torch.jit.script(payload), str(ts_path))
    except Exception as exc:           # noqa: BLE001
        print("Could not TorchScript-export anomalib model:", exc)
    return ts_path

if "pc" in locals():
    export_patchcore(pc, MODELS_DIR)


In [ ]:
"""CPU latency baseline on the dev machine (Jetson INT8 will be 3-5x faster)."""
def benchmark_onnx(onnx_path: Path, input_shape=(1, 3, 640, 640), iters: int = 50) -> float:
    """Return mean inference latency in ms over `iters` runs (CPU EP)."""
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    inp = np.random.randn(*input_shape).astype(np.float32)
    name = sess.get_inputs()[0].name
    for _ in range(5):                       # warm-up
        sess.run(None, {name: inp})
    t0 = time.perf_counter()
    for _ in range(iters):
        sess.run(None, {name: inp})
    return (time.perf_counter() - t0) * 1000 / iters

yolo_onnx = MODELS_DIR / "yolo11n_seg.onnx"
if yolo_onnx.exists():
    ms = benchmark_onnx(yolo_onnx, (1, 3, IMG_SIZE, IMG_SIZE))
    print(f"YOLOv11n-seg CPU latency: {ms:.1f} ms ({1000/ms:.1f} FPS)")
    print("Expected on Jetson Orin Nano @ INT8: ~3-5x faster (-> 30-50 FPS feature, ~10-15 FPS end-to-end).")
else:
    print("No ONNX yet - run the export cell above.")


### 4.1 On-device vs Cloud-VLM cost comparison

This table is filled with vendor-published figures (Q1 2026). It is a
*framework* for the trade-off discussion in the seminar report; numbers
should be re-checked closer to deployment.

| Backend | Latency / frame | Cost / frame | Notes |
|---------|-----------------|--------------|-------|
| Jetson Orin Nano + YOLOv11n-seg INT8 | ~70 ms (~14 FPS) | electricity only | offline, deterministic, no PII leaves the line |
| GPT-4V (`gpt-4o`) via API | 800-2000 ms | ~0.005 USD/img | needs internet, rate-limited, opaque updates |
| Claude 4 Vision (Sonnet) | 700-1500 ms | ~0.003 USD/img | same constraints as GPT-4V |
| Cloud GPU (T4) hosting our own YOLO | ~25 ms | ~0.0001 USD/img | needs internet + ops overhead |

**Conclusion:** for a fixed-purpose 9-class inspection task running 24/7
on a manufacturing line, the on-device pipeline is ~50-1000x cheaper per
frame and removes the network as a single point of failure. A cloud VLM
remains attractive only for *new* defect types where flexible reasoning
is more valuable than throughput.


## 5. Conversion Notes for Jetson

**Primary deployment:** use `scripts/jetson_inference.py` directly. It is
a standalone, dependency-light Python script (TensorRT + pycuda + opencv
only) that does NOT need nbconvert and is the recommended entry point on
the Jetson.

**Fallback:** if you ever need to derive a script from this notebook,
cells are properly tagged `skip-export` (see `tools/tag_cells.py`) so:

```bash
jupyter nbconvert --to script kip_inspection.ipynb \
                  --TagRemovePreprocessor.remove_cell_tags='{"skip-export"}'
```

The result is `kip_inspection.py`. The ready-made deployment script is
[`scripts/jetson_inference.py`](../scripts/jetson_inference.py); copy it
together with the exported engines:

```bash
# on the dev machine
scp models/yolo11n_seg.onnx       jetson:~/kip/models/
scp models/patchcore_resnet18.onnx jetson:~/kip/models/
scp scripts/export_to_jetson.sh   jetson:~/kip/
scp scripts/jetson_inference.py   jetson:~/kip/

# on the Jetson
cd ~/kip
bash export_to_jetson.sh ./models
python jetson_inference.py \
    --yolo models/yolo11n_seg.int8.engine \
    --patchcore models/patchcore_resnet18.fp16.engine \
    --camera 0
```

The cell below is the `# %% [export-as-script]` entry point - it gets
the `if __name__ == "__main__"` guard so the converted script is
runnable end-to-end on the Jetson.


In [ ]:
# %% [export-as-script]
"""Real-time pipeline for direct execution as a converted Python script."""
def run_realtime_pipeline(camera_id: int = 0,
                          yolo_engine: "YOLO | None" = None,
                          patchcore_engine=None,
                          conf: float = 0.25,
                          defect_threshold: float = 0.5) -> None:
    """Run YOLO + PatchCore on a webcam feed; press 'q' to quit."""
    if yolo_engine is None:
        yolo_engine = YOLO(str(MODELS_DIR / "yolo11n_seg.onnx"))

    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open camera {camera_id}")

    fps_t0, fps_n, fps = time.time(), 0, 0.0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            res = yolo_engine.predict(source=frame, imgsz=IMG_SIZE,
                                       conf=conf, device=DEVICE, verbose=False)[0]
            if res.boxes is not None and len(res.boxes) > 0:
                xyxys = res.boxes.xyxy.cpu().numpy().astype(int)
                clses = res.boxes.cls.cpu().numpy().astype(int)
                confs = res.boxes.conf.cpu().numpy()
                for (xyxy, cls, sc) in zip(xyxys, clses, confs):
                    x1, y1, x2, y2 = xyxy
                    crop = frame[y1:y2, x1:x2]
                    is_defect, anom = False, 0.0
                    if patchcore_engine is not None and crop.size > 0:
                        anom, _, is_defect = detect_defect(
                            crop, patchcore_engine, threshold=defect_threshold)
                    color = (0, 0, 255) if is_defect else (0, 200, 0)
                    label = f"{CLASS_NAMES[cls]} {sc:.2f}"
                    if patchcore_engine is not None:
                        label += f" | a={anom:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, label, (x1, max(15, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

            fps_n += 1
            if fps_n >= 10:
                fps = fps_n / (time.time() - fps_t0)
                fps_t0, fps_n = time.time(), 0
            cv2.putText(frame, f"FPS: {fps:.1f}", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)

            cv2.imshow("KIP Inspection", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    from ultralytics import YOLO
    yolo = YOLO(str(MODELS_DIR / "yolo11n_seg.onnx"))
    run_realtime_pipeline(camera_id=0, yolo_engine=yolo, patchcore_engine=None)
